<a href="https://colab.research.google.com/github/VitorTardivo21/Redes-Neurais-e-IA-Aplicada/blob/main/02_limpeza_textos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!git clone https://github.com/VitorTardivo21/Redes-Neurais-e-IA-Aplicada.git

Cloning into 'Redes-Neurais-e-IA-Aplicada'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 113 (delta 47), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 46.48 KiB | 1.33 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [ ]:
!pip install pypdf requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 2.0 MB/s eta 0:00:00


In [6]:
"""
Etapa 2 — Pipeline de Extração, Limpeza e Estruturação
Projeto: Diário Oficial Inteligente de Avaré

Atividade 1 — Extrair texto dos PDFs com pdfplumber
Atividade 2 — Limpar e normalizar os textos
Atividade 3 — Criar campos estruturados e montar a base
Atividade 4 — Gerar amostra rotulada automaticamente
Atividade 5 — Salvar em CSV e SQLite na estrutura de pastas do projeto
"""

import os
import re
import sqlite3
import pandas as pd

from src.extract_text import extrair_de_pdf
from src.preprocess   import processar

# ==========================================
# CAMINHOS
# ==========================================

BASE_DIR   = os.path.dirname(os.path.abspath(__file__))

# PDFs baixados na Etapa 1 (equivalente a data/raw/ na estrutura recomendada)
PASTA_PDFS = os.path.join(BASE_DIR, "pdf")

# Metadados coletados na Etapa 1
CSV_ETAPA1 = os.path.join(BASE_DIR, "diarios_avare.csv")

# Saídas da Etapa 2
PASTA_PROCESSED    = os.path.join(BASE_DIR, "data", "processed")
CSV_BASE_TEXTUAL   = os.path.join(PASTA_PROCESSED, "base_textual.csv")
CSV_AMOSTRA        = os.path.join(PASTA_PROCESSED, "amostra_rotulada.csv")
DB_SQLITE          = os.path.join(PASTA_PROCESSED, "diario_avare.db")

# ==========================================
# ATIVIDADE 3 — CAMPOS ESTRUTURADOS
# Detecção automática de tipo_ato, título e secretaria
# ==========================================

# Ordem de prioridade: cada lista é testada de cima para baixo
TIPO_ATO_PADROES = [
    ("contas_publicas", [
        r"relatório\s+resumido",
        r"execução\s+orçamentária",
        r"balanço\s+orçamentário",
        r"RGF\s*[–\-]",
        r"RREO\s*[–\-]",
        r"balancete",
        r"prestação\s+de\s+contas",
        r"demonstrativo\s+da\s+receita",
        r"demonstrativo\s+das\s+despesas",
        r"gestão\s+fiscal",
    ]),
    ("edital_concurso", [
        r"concurso\s+público",
        r"processo\s+seletivo",
        r"inscrições\s+abertas",
        r"edital\s+de\s+concurso",
        r"edital\s+de\s+processo\s+seletivo",
    ]),
    ("licitacao_contrato", [
        r"pregão\s+eletrônico",
        r"pregão\s+presencial",
        r"ata\s+de\s+registro\s+de\s+preços",
        r"extrato\s+de\s+(?:contrato|ata|termo\s+aditivo)",
        r"empresa\s+vencedora",
        r"aviso\s+de\s+(?:licitação|edital)",
        r"processo\s+licitatório",
        r"dispensa\s+de\s+licitação",
        r"inexigibilidade",
        r"homologação\s+de\s+(?:licitação|pregão)",
    ]),
    ("decreto", [
        r"decreto\s+(?:municipal\s+)?n[.º°]?\s*[\d.]+",
        r"fica\s+decretado",
        r"usando\s+das\s+suas\s+atribuições",
    ]),
    ("portaria", [
        r"portaria\s+n[.º°]?\s*[\d.]+",
        r"resolve\s+expedir",
        r"resolve\s+determinar",
    ]),
    ("ato_pessoal", [
        r"\bnomear\b",
        r"\bexonerar\b",
        r"\bnomeação\b",
        r"\bexoneração\b",
        r"cargo\s+em\s+comissão",
        r"servidor\s+(?:público|municipal)",
        r"\bempossado\b",
        r"posse\s+e\s+exercício",
    ]),
]

# Padrões prioritários por tipo de ato — tentados antes da lista geral
TITULO_POR_TIPO = {
    "decreto": [
        r"(Decreto(?:\s+Municipal)?\s+n[.°º\s]*[\d.,]+,\s+de\s+\d+\s+de\s+\w+\s+de\s+\d{4}[^)\n]*\)?\.?)",
    ],
    "portaria": [
        r"(Portaria\s+n[.°º\s]*[\d.,]+[^\n]*)",
    ],
    "licitacao_contrato": [
        r"(AVISO\s+DE\s+[A-ZÁÉÍÓÚÃÕÂÊÔÇ][A-ZÁÉÍÓÚÃÕÂÊÔÇ\s]+)",
        r"(EXTRATO\s+DE\s+[A-ZÁÉÍÓÚÃÕÂÊÔÇ][A-ZÁÉÍÓÚÃÕÂÊÔÇ\s]+)",
        r"(PREGÃO\s+[^\n]+)",
        r"(HOMOLOGA[CÇ][AÃ]O[^\n]*)",
    ],
    "edital_concurso": [
        r"(CONCURSO\s+PÚBLICO\s*[^\n]*)",
        r"(PROCESSO\s+SELETIVO\s*[^\n]*)",
        r"(EDITAL\s+[^\n]*)",
    ],
    "contas_publicas": [
        r"(RELATÓRIO\s+[^\n.]{5,80})",       # qualquer char exceto nova linha e ponto
        r"(DEMONSTRATIVO\s+[^\n.]{5,80})",
        r"(BALANÇO\s+[^\n.]{5,60})",
        r"(RGF\s*[-–]\s*ANEXO\s*\d+[^\n]*)",  # ex: RGF – ANEXO 4
        r"(RREO\s*[-–]\s*ANEXO\s*\d+[^\n]*)", # ex: RREO – Anexo 8
    ],
}

# Lista geral usada como fallback quando o tipo não tem padrão específico
TITULO_PADROES = [
    r"(Decreto(?:\s+Municipal)?\s+n[.°º\s]*[\d.,]+,\s+de\s+\d+\s+de\s+\w+\s+de\s+\d{4}[^)\n]*\)?\.?)",
    r"(Portaria\s+n[.°º\s]*[\d.,]+[^\n]*)",
    r"(AVISO\s+DE\s+[A-ZÁÉÍÓÚÃÕÂÊÔÇ][A-ZÁÉÍÓÚÃÕÂÊÔÇ\s]+)",
    r"(EXTRATO\s+DE\s+[A-ZÁÉÍÓÚÃÕÂÊÔÇ][A-ZÁÉÍÓÚÃÕÂÊÔÇ\s]+)",
    r"(CONCURSO\s+PÚBLICO\s*[^\n]*)",
    r"(EDITAL\s+[^\n]*)",
    r"(CONVOCAÇÃO[^\n]*)",
    r"(COMUNICADO[^\n]*)",
    r"(RELATÓRIO\s+[A-ZÁÉÍÓÚÃÕÂÊÔÇ\s]+)",
    r"(DEMONSTRATIVO\s+[A-ZÁÉÍÓÚÃÕÂÊÔÇ\s]+)",
]

# Linhas de cabeçalho do SEMANÁRIO a ignorar no fallback do título
LINHAS_CABECALHO = re.compile(
    r"^("
    r"SEMANÁRIO|Oficial Eletrônico|avare\.sp\.gov\.br"
    r"|(?:Sexta|Segunda|Terça|Quarta|Quinta)-feira,\s.*"   # linha completa de data
    r"|PODER EXECUTIVO|Poder Executivo|Atos Oficiais|Decretos?"
    r")$",
    re.IGNORECASE,
)
# Palavras-chave que indicam linha de cabeçalho mesmo sem match exato
PALAVRAS_CABECALHO = ("Prefeito:", "Edição nº", "Edição n.")

SECRETARIA_PADROES = [
    # Busca apenas no início do texto (antes de entrar no corpo do ato)
    r"(Secretaria\s+Municipal\s+(?:de|da|do)\s+[A-ZÁÉÍÓÚ][^\n,;.]{3,60})",
    r"(Secretaria\s+(?:de|da|do)\s+[A-ZÁÉÍÓÚ][^\n,;.]{3,60})",
    r"(CÂMARA\s+MUNICIPAL[^\n]*)",
    r"(FREA\s*[-–]?\s*Fundação[^\n]*)",
]


def detectar_tipo_ato(texto: str) -> str:
    texto_lower = texto.lower()
    for tipo, padroes in TIPO_ATO_PADROES:
        for padrao in padroes:
            if re.search(padrao, texto_lower):
                return tipo
    return "outros"


def extrair_titulo(texto: str, tipo_ato: str = "") -> str:
    # Tenta os padrões específicos do tipo_ato primeiro
    padroes = list(TITULO_POR_TIPO.get(tipo_ato, [])) + TITULO_PADROES
    for padrao in padroes:
        m = re.search(padrao, texto, re.IGNORECASE)
        if m:
            return m.group(1).strip()[:200]
    # Fallback: primeira linha relevante que não seja do cabeçalho do jornal
    for linha in texto.splitlines():
        linha = linha.strip()
        if (len(linha) >= 10
                and not LINHAS_CABECALHO.match(linha)
                and not any(p in linha for p in PALAVRAS_CABECALHO)):
            return linha[:200]
    return "Sem título"


def detectar_secretaria(texto: str) -> str:
    # Limita a busca ao cabeçalho organizacional (primeiros 400 chars)
    cabecalho = texto[:400]
    for padrao in SECRETARIA_PADROES:
        m = re.search(padrao, cabecalho, re.IGNORECASE)
        if m:
            return m.group(1).strip()[:100]
    if re.search(r"câmara\s+municipal|poder\s+legislativo", cabecalho, re.IGNORECASE):
        return "Câmara Municipal de Avaré"
    return "Prefeitura Municipal de Avaré"


def criar_registro(seq, data, edicao, pagina, total_pag, tipo, titulo, secretaria, texto, url):
    ano = str(data)[:4]
    return {
        "id":              f"DOA-{ano}-{seq:04d}",
        "data_publicacao": data,
        "numero_edicao":   edicao,
        "pagina":          pagina,
        "paginas_total":   total_pag,
        "tipo_ato":        tipo,
        "titulo":          titulo,
        "secretaria":      secretaria,
        "texto":           texto,
        "url_original":    url,
        "rotulo":          "",   # preenchido na Atividade 4
    }


# ==========================================
# EXECUÇÃO PRINCIPAL
# ==========================================

print("=" * 50)
print("ETAPA 2 — Iniciando pipeline")
print("=" * 50)

# Carrega metadados da Etapa 1
print("\n[1/5] Carregando metadados da Etapa 1...")
df_meta = pd.read_csv(CSV_ETAPA1, encoding="utf-8-sig")
edicoes = df_meta.drop_duplicates("edicao")[["edicao", "data", "paginas_total", "url"]].copy()
print(f"      {len(edicoes)} edições para processar")

registros       = []
seq             = 1
pdfs_escaneados = []
pags_branco     = 0

# -------------------------------------------------------
# ATIVIDADE 1 + 2: Extração, limpeza e estruturação
# -------------------------------------------------------
print("\n[2/5] Extraindo e processando texto dos PDFs...")

for _, ed in edicoes.iterrows():
    edicao  = str(ed["edicao"])
    data    = ed["data"]
    url     = ed["url"]
    total   = ed["paginas_total"]
    ano, mes = data[:4], data[5:7]

    caminho_pdf = os.path.join(PASTA_PDFS, ano, mes, f"{data}_{edicao}.pdf")

    if not os.path.exists(caminho_pdf):
        print(f"  [AVISO] PDF não encontrado: {caminho_pdf}")
        continue

    print(f"  Edição {edicao} ({data})...", end=" ")

    # --- Atividade 1: Extração ---
    textos_por_pagina, escaneado = extrair_de_pdf(caminho_pdf)

    if escaneado:
        print("ESCANEADO — sem texto extraível")
        pdfs_escaneados.append({"edicao": edicao, "data": data})
        continue

    paginas_extraidas = 0

    for num_pag, texto_bruto in enumerate(textos_por_pagina, start=1):
        if not texto_bruto.strip():
            pags_branco += 1
            continue

        # --- Atividade 2: Limpeza e normalização ---
        texto_limpo = processar(texto_bruto)

        if not texto_limpo:
            pags_branco += 1
            continue

        # --- Atividade 3: Campos estruturados ---
        tipo       = detectar_tipo_ato(texto_limpo)
        titulo     = extrair_titulo(texto_limpo, tipo)   # usa padrões do tipo_ato
        secretaria = detectar_secretaria(texto_limpo)

        registros.append(
            criar_registro(seq, data, edicao, num_pag, total,
                           tipo, titulo, secretaria, texto_limpo, url)
        )
        seq += 1
        paginas_extraidas += 1

    print(f"{paginas_extraidas} páginas extraídas")

# -------------------------------------------------------
# ATIVIDADE 4: Rotulagem automática
# -------------------------------------------------------
print("\n[3/5] Aplicando rótulos automáticos...")

df = pd.DataFrame(registros)

# O rótulo usa os mesmos critérios do tipo_ato — aqui fica explícito para
# o campo 'rotulo' que será usado no treinamento supervisionado (Etapa 4).
# ATENÇÃO: rótulos automáticos devem ser revisados manualmente antes de treinar.
df["rotulo"] = df["tipo_ato"]

dist = df["rotulo"].value_counts()
print("  Distribuição de rótulos:")
for classe, n in dist.items():
    print(f"    {classe:<22} {n:>4} registros")

# -------------------------------------------------------
# ATIVIDADE 5a: Salvar base completa
# -------------------------------------------------------
print("\n[4/5] Salvando base textual...")
df.to_csv(CSV_BASE_TEXTUAL, index=False, encoding="utf-8-sig")
print(f"      {len(df)} registros ->{CSV_BASE_TEXTUAL}")

# Salvar também em SQLite
conn = sqlite3.connect(DB_SQLITE)
df.to_sql("publicacoes", conn, if_exists="replace", index=False)
conn.close()
print(f"      SQLite ->{DB_SQLITE}")

# -------------------------------------------------------
# ATIVIDADE 5b: Amostra rotulada (mínimo 40 registros, ≥3 classes)
# -------------------------------------------------------
print("\n[5/5] Gerando amostra rotulada...")

MAX_POR_CLASSE = 10
frames = []

for classe in df["rotulo"].unique():
    subset = df[df["rotulo"] == classe].head(MAX_POR_CLASSE)
    frames.append(subset)

df_amostra = pd.concat(frames).reset_index(drop=True)

# Garante o mínimo de 40 mesmo se houver poucas classes
if len(df_amostra) < 40:
    faltam = 40 - len(df_amostra)
    extras = df[~df["id"].isin(df_amostra["id"])].head(faltam)
    df_amostra = pd.concat([df_amostra, extras]).reset_index(drop=True)

df_amostra.to_csv(CSV_AMOSTRA, index=False, encoding="utf-8-sig")

print(f"      {len(df_amostra)} registros na amostra ({df_amostra['rotulo'].nunique()} classes)")
print(f"      ->{CSV_AMOSTRA}")

# ==========================================
# RELATÓRIO FINAL
# ==========================================

print("\n" + "=" * 50)
print("ETAPA 2 — CONCLUÍDA")
print("=" * 50)
print(f"Páginas processadas       : {len(df)}")
print(f"Páginas em branco ignoradas: {pags_branco}")
print(f"PDFs escaneados detectados : {len(pdfs_escaneados)}")

if pdfs_escaneados:
    print("  (requerem OCR — ver src/extract_text.py)")
    for p in pdfs_escaneados:
        print(f"    Edição {p['edicao']} — {p['data']}")

print(f"\nArquivos gerados:")
print(f"  {CSV_BASE_TEXTUAL}")
print(f"  {CSV_AMOSTRA}")
print(f"  {DB_SQLITE}")

print(f"\nTamanho médio do texto por página: {df['texto'].str.len().mean():.0f} chars")

print("\nExemplo de registro:")
ex = df.iloc[0]
print(f"  id          : {ex['id']}")
print(f"  data        : {ex['data_publicacao']}")
print(f"  edicao      : {ex['numero_edicao']}")
print(f"  tipo_ato    : {ex['tipo_ato']}")
print(f"  titulo      : {ex['titulo'][:80]}")
print(f"  secretaria  : {ex['secretaria']}")
print(f"  rotulo      : {ex['rotulo']}")
print(f"  texto[:200] : {ex['texto'][:200]}")

ModuleNotFoundError: No module named 'src'